# Query Game Events from Database

This notebook queries a specific game from the PostgreSQL database and displays all its events.

## Setup

In [26]:
import psycopg2
import pandas as pd
from IPython.display import display

In [27]:
# Database connection configuration
DB_CONFIG = {
    'dbname': 'ufa_analytics',
    'user': os.getenv('DB_USER', 'postgres'),
    'password': '',
    'host': 'localhost',
    'port': 5432
}

def get_connection():
    """Create and return a database connection."""
    return psycopg2.connect(**DB_CONFIG)

## Configuration: Set Game ID

In [28]:
# Set the game ID you want to query
GAME_ID = "2024-04-27-MTL-NY"  # Change this to any game ID in your database

## Query Game Details

In [29]:
conn = get_connection()
cur = conn.cursor()

# Get game details
cur.execute("""
    SELECT game_id, home_team_id, away_team_id, home_score, away_score, game_date, year
    FROM games
    WHERE game_id = %s;
""", (GAME_ID,))

game = cur.fetchone()

if game:
    print("="*80)
    print(f"GAME: {game[0]}")
    print("="*80)
    print(f"Date: {game[5]}")
    print(f"Teams: {game[1]} (Home) vs {game[2]} (Away)")
    print(f"Final Score: {game[3]} - {game[4]}")
    print(f"Year: {game[6]}")
    print("="*80)
else:
    print(f"❌ Game {GAME_ID} not found in database!")
    cur.close()
    conn.close()

cur.close()
conn.close()

GAME: 2024-04-27-MTL-NY
Date: 2024-04-27
Teams: empire (Home) vs royal (Away)
Final Score: 18 - 16
Year: 2024


## Query All Events for This Game

In [30]:
conn = get_connection()

# Query all events
query = """
    SELECT 
        event_id,
        event_number,
        event_type,
        team,
        year,
        time,
        puller,
        pull_x,
        pull_y,
        pull_ms,
        thrower,
        thrower_x,
        thrower_y,
        receiver,
        receiver_x,
        receiver_y,
        turnover_x,
        turnover_y,
        defender,
        player,
        synthetic
    FROM events
    WHERE game_id = %s
    ORDER BY event_number;
"""

events_df = pd.read_sql_query(query, conn, params=(GAME_ID,))

conn.close()

print(f"\nTotal events retrieved: {len(events_df)}")
print(f"Synthetic events: {events_df['synthetic'].sum()}")
print(f"\nFirst 10 events:")
display(events_df.head(10))


Total events retrieved: 675
Synthetic events: 1

First 10 events:


/var/folders/pm/ldzj5gjs287bk95mxn8gjznc0000gn/T/ipykernel_48071/2206935231.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  events_df = pd.read_sql_query(query, conn, params=(GAME_ID,))


,event_id,event_number,event_type,team,year,time,puller,pull_x,pull_y,pull_ms,...,thrower_x,thrower_y,receiver,receiver_x,receiver_y,turnover_x,turnover_y,defender,player,synthetic
0,1417,1,1,royal,2024,0.0,None,NaN,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,False
1,1418,2,2,empire,2024,0.0,None,NaN,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,False
2,1419,3,7,royal,2024,NaN,ctremblay,-16.47,89.19,3152.0,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,None,False
3,1420,4,18,empire,2024,NaN,None,NaN,NaN,NaN,...,22.64,7.09,srueschem,4.63,9.40,NaN,NaN,None,None,False
4,1421,5,18,empire,2024,NaN,None,NaN,NaN,NaN,...,4.63,9.40,ochartock,-13.51,18.61,NaN,NaN,None,None,False
5,1422,6,22,empire,2024,NaN,None,NaN,NaN,NaN,...,-13.51,18.61,None,NaN,NaN,-23.4,38.31,None,None,False
6,1423,7,11,royal,2024,NaN,None,NaN,NaN,NaN,...,NaN,NaN,None,NaN,NaN,NaN,NaN,ctremblay,None,False
7,1424,8,18,royal,2024,NaN,None,NaN,NaN,NaN,...,-23.61,62.11,ctremblay,-11.08,67.35,NaN,NaN,None,None,False
8,1425,9,18,royal,2024,NaN,None,NaN,NaN,NaN,...,-11.08,67.35,rlalondel,-4.45,68.37,NaN,NaN,None,None,False
9,1426,10,18,royal,2024,NaN,None,NaN,NaN,NaN,...,-4.45,68.37,cguay,-17.49,68.73,NaN,NaN,None,None,False


## Event Type Breakdown

In [31]:
# Event type mapping
event_type_names = {
    1: "Starting D",
    2: "Starting O",
    3: "Timeout",
    7: "Pull (Inbounds)",
    8: "Pull (Out of Bounds)",
    11: "Block",
    18: "Pass",
    19: "Goal",
    20: "Drop",
    21: "Dropped Pull",
    22: "Throwaway",
    23: "Callahan",
    24: "Stall",
    25: "Injury",
    26: "Player Misconduct",
    27: "Player Ejected",
    28: "End Q1",
    29: "End Q2",
    30: "End Q3",
    31: "End Q4",
    32: "End OT1",
    33: "End OT2"
}

# Add event type names to dataframe
events_df['event_type_name'] = events_df['event_type'].map(event_type_names)

# Count by event type
event_counts = events_df.groupby(['event_type', 'event_type_name']).size().reset_index(name='count')
event_counts = event_counts.sort_values('event_type')

print("\nEvent Type Breakdown:")
print("="*60)
for _, row in event_counts.iterrows():
    print(f"{row['event_type_name']:25s}: {row['count']:4d} events")
print("="*60)
print(f"{'TOTAL':25s}: {len(events_df):4d} events")


Event Type Breakdown:
Starting D               :   37 events
Starting O               :   37 events
Timeout                  :    6 events
Pull (Inbounds)          :   31 events
Pull (Out of Bounds)     :    6 events
Block                    :   29 events
Pass                     :  437 events
Goal                     :   34 events
Drop                     :    6 events
Throwaway                :   42 events
Stall                    :    1 events
Injury                   :    1 events
End Q1                   :    2 events
End Q2                   :    2 events
End Q3                   :    2 events
End Q4                   :    2 events
TOTAL                    :  675 events


## Filter Events by Type

Examples of useful filters:

In [32]:
# Get all goals (type 19)
goals = events_df[events_df['event_type'] == 19][['event_number', 'team', 'thrower', 'receiver']]
print(f"\nGoals ({len(goals)} total):")
display(goals)


Goals (34 total):


,event_number,team,thrower,receiver
35,37,empire,cweinberg,bjagt
49,52,royal,kquinlan,tdecraene
69,74,empire,skeegan,bjagt
99,107,royal,kgroulx,plebourda
118,130,empire,ochartock,jwilliams
130,144,empire,efortin,jrandolph
147,162,royal,tdecraene,kgroulx
160,178,empire,srueschem,cweinberg
203,223,royal,tdecraene,kquinlan
209,230,empire,lhaberfie,jwilliams


In [33]:
# Get all turnovers (types 20, 22, 23, 24)
turnovers = events_df[events_df['event_type'].isin([20, 22, 23, 24])][[
    'event_number', 'event_type_name', 'team', 'thrower', 'turnover_x', 'turnover_y', 'synthetic'
]]
print(f"\nTurnovers ({len(turnovers)} total):")
display(turnovers)


Turnovers (49 total):


,event_number,event_type_name,team,thrower,turnover_x,turnover_y,synthetic
5,6,Throwaway,empire,ochartock,-23.400000,38.310000,False
11,12,Throwaway,royal,rlalondel,-22.230000,75.650000,False
15,16,Throwaway,empire,cweinberg,-12.210000,59.520000,False
17,19,Throwaway,royal,ctremblay,12.310000,110.750000,False
20,22,Throwaway,empire,jwilliams,11.590000,69.200000,False
26,28,Throwaway,royal,tlalondel,4.880000,107.250000,False
41,44,Throwaway,royal,dlacombeb,-15.600000,25.120000,False
46,49,Throwaway,empire,jrandolph,-0.140000,101.720000,False
55,59,Throwaway,empire,lhaberfie,-0.690000,96.540000,False
61,65,Drop,royal,rlalondel,NaN,NaN,False


In [34]:
# Get all blocks (type 11)
blocks = events_df[events_df['event_type'] == 11][['event_number', 'team', 'defender']]
print(f"\nBlocks ({len(blocks)} total):")
display(blocks)


Blocks (29 total):


,event_number,team,defender
6,7,royal,ctremblay
12,13,empire,ochartock
18,20,empire,cweinberg
21,23,royal,ctremblay
27,29,empire,lhaberfie
42,45,empire,mbrownlee
47,50,royal,kquinlan
56,60,royal,swarrick
106,115,royal,ctremblay
166,185,empire,jholm


## Get All Events as Dictionary (Original Format)

This returns events in the same format as the API data:

In [36]:
# Convert DataFrame to list of dictionaries (similar to original API format)
events_list = []

for _, row in events_df.iterrows():
    event = {
        'type': row['event_type'],
        'team': row['team'],
        'year': row['year'],
        'gameID': GAME_ID
    }
    
    # Add fields that exist
    if pd.notna(row['time']):
        event['time'] = int(row['time'])
    if pd.notna(row['puller']):
        event['puller'] = row['puller']
    if pd.notna(row['pull_x']):
        event['pullX'] = row['pull_x']
    if pd.notna(row['pull_y']):
        event['pullY'] = row['pull_y']
    if pd.notna(row['pull_ms']):
        event['pullMs'] = int(row['pull_ms'])
    if pd.notna(row['thrower']):
        event['thrower'] = row['thrower']
    if pd.notna(row['thrower_x']):
        event['throwerX'] = row['thrower_x']
    if pd.notna(row['thrower_y']):
        event['throwerY'] = row['thrower_y']
    if pd.notna(row['receiver']):
        event['receiver'] = row['receiver']
    if pd.notna(row['receiver_x']):
        event['receiverX'] = row['receiver_x']
    if pd.notna(row['receiver_y']):
        event['receiverY'] = row['receiver_y']
    if pd.notna(row['turnover_x']):
        event['turnoverX'] = row['turnover_x']
    if pd.notna(row['turnover_y']):
        event['turnoverY'] = row['turnover_y']
    if pd.notna(row['defender']):
        event['defender'] = row['defender']
    if pd.notna(row['player']):
        event['player'] = row['player']
    if row['synthetic']:
        event['synthetic'] = True
    
    events_list.append(event)

print(f"\nConverted {len(events_list)} events to dictionary format")
for i, event in enumerate(events_list):
    print(f"\n{i}. Type {event['type']} - {event_type_names.get(event['type'], 'Unknown')}")
    print(f"   {event}")


Converted 675 events to dictionary format

0. Type 1 - Starting D
   {'type': 1, 'team': 'royal', 'year': 2024, 'gameID': '2024-04-27-MTL-NY', 'time': 0}

1. Type 2 - Starting O
   {'type': 2, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY', 'time': 0}

2. Type 7 - Pull (Inbounds)
   {'type': 7, 'team': 'royal', 'year': 2024, 'gameID': '2024-04-27-MTL-NY', 'puller': 'ctremblay', 'pullX': -16.47, 'pullY': 89.19, 'pullMs': 3152}

3. Type 18 - Pass
   {'type': 18, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY', 'thrower': 'skeegan', 'throwerX': 22.64, 'throwerY': 7.09, 'receiver': 'srueschem', 'receiverX': 4.63, 'receiverY': 9.4}

4. Type 18 - Pass
   {'type': 18, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY', 'thrower': 'srueschem', 'throwerX': 4.63, 'throwerY': 9.4, 'receiver': 'ochartock', 'receiverX': -13.51, 'receiverY': 18.61}

5. Type 22 - Throwaway
   {'type': 22, 'team': 'empire', 'year': 2024, 'gameID': '2024-04-27-MTL-NY', 'thrower'

## Save Events to Variable

The `events_list` variable now contains all events for this game in dictionary format, ready for further analysis or processing!